In [1]:
import pandas as pd

feature_path = "../data/processed/periyar_rainfall_features_2015.csv"

df = pd.read_csv(
    feature_path,
    parse_dates=["time"]
)

print("Shape:", df.shape)
print()
print(df.head())

Shape: (351, 8)

        time  rainfall_mm  rainfall_3day  rainfall_7day  rainfall_15day  \
0 2015-01-15          0.0            0.0       0.925609        2.209708   
1 2015-01-16          0.0            0.0       0.437691        0.925609   
2 2015-01-17          0.0            0.0       0.000000        0.925609   
3 2015-01-18          0.0            0.0       0.000000        0.925609   
4 2015-01-19          0.0            0.0       0.000000        0.925609   

   rainfall_3day_max  rainfall_7day_max  rainfall_previous_day  
0                0.0           0.487918                    0.0  
1                0.0           0.437691                    0.0  
2                0.0           0.000000                    0.0  
3                0.0           0.000000                    0.0  
4                0.0           0.000000                    0.0  


In [2]:
print(df.isnull().sum())

time                     0
rainfall_mm              0
rainfall_3day            0
rainfall_7day            0
rainfall_15day           0
rainfall_3day_max        0
rainfall_7day_max        0
rainfall_previous_day    0
dtype: int64


In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/periyar_rainfall_features_full.csv", parse_dates=["time"])
df = df.sort_values("time").reset_index(drop=True)

feature_cols = [
    "rainfall_mm", "rainfall_3day", "rainfall_7day", "rainfall_15day",
    "rainfall_3day_max", "rainfall_7day_max", "rainfall_previous_day"
]
target_col = "target_heavy_rain_next_day"

train = df[df["time"] < "2022-01-01"]
test = df[df["time"] >= "2022-01-01"]

X_train, y_train = train[feature_cols], train[target_col]
X_test, y_test = test[feature_cols], test[target_col]

print("Train:", X_train.shape, "Heavy rain days:", y_train.sum())
print("Test:", X_test.shape, "Heavy rain days:", y_test.sum())

Train: (2543, 7) Heavy rain days: 29
Test: (730, 7) Heavy rain days: 7


In [2]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_persistence = (test["rainfall_mm"] >= 64.5).astype(int)

print("Persistence baseline:")
print(confusion_matrix(y_test, y_pred_persistence))
print(classification_report(y_test, y_pred_persistence, digits=3))

Persistence baseline:
[[719   4]
 [  4   3]]
              precision    recall  f1-score   support

           0      0.994     0.994     0.994       723
           1      0.429     0.429     0.429         7

    accuracy                          0.989       730
   macro avg      0.712     0.712     0.712       730
weighted avg      0.989     0.989     0.989       730



In [3]:
import lightgbm as lgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    num_leaves=15,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("LightGBM:")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

scale_pos_weight: 86.6896551724138
[LightGBM] [Info] Number of positive: 29, number of negative: 2514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000334 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1775
[LightGBM] [Info] Number of data points in the train set: 2543, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.011404 -> initscore=-4.462335
[LightGBM] [Info] Start training from score -4.462335
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

In [4]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance)

rainfall_mm              486
rainfall_15day           485
rainfall_7day            333
rainfall_previous_day    333
rainfall_7day_max        276
rainfall_3day            270
rainfall_3day_max        163
dtype: int32


In [5]:
import numpy as np

def csi(y_true, y_pred):
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    return tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0

for threshold in [0.5, 0.4, 0.3, 0.2, 0.15, 0.1, 0.05]:
    y_pred_t = (y_proba >= threshold).astype(int)
    tp = ((y_test == 1) & (y_pred_t == 1)).sum()
    fp = ((y_test == 0) & (y_pred_t == 1)).sum()
    fn = ((y_test == 1) & (y_pred_t == 0)).sum()
    print(f"threshold={threshold:.2f}  TP={tp} FP={fp} FN={fn}  CSI={csi(y_test, y_pred_t):.3f}")

threshold=0.50  TP=3 FP=14 FN=4  CSI=0.143
threshold=0.40  TP=3 FP=15 FN=4  CSI=0.136
threshold=0.30  TP=3 FP=16 FN=4  CSI=0.130
threshold=0.20  TP=3 FP=23 FN=4  CSI=0.100
threshold=0.15  TP=3 FP=26 FN=4  CSI=0.091
threshold=0.10  TP=4 FP=29 FN=3  CSI=0.111
threshold=0.05  TP=4 FP=45 FN=3  CSI=0.077


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(class_weight="balanced", max_iter=1000)
logreg.fit(X_train_s, y_train)
y_proba_lr = logreg.predict_proba(X_test_s)[:, 1]

for threshold in [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3]:
    y_pred_t = (y_proba_lr >= threshold).astype(int)
    tp = ((y_test == 1) & (y_pred_t == 1)).sum()
    fp = ((y_test == 0) & (y_pred_t == 1)).sum()
    fn = ((y_test == 1) & (y_pred_t == 0)).sum()
    print(f"threshold={threshold:.2f}  TP={tp} FP={fp} FN={fn}  CSI={csi(y_test, y_pred_t):.3f}")

threshold=0.90  TP=3 FP=19 FN=4  CSI=0.115
threshold=0.80  TP=3 FP=34 FN=4  CSI=0.073
threshold=0.70  TP=4 FP=48 FN=3  CSI=0.073
threshold=0.60  TP=6 FP=56 FN=1  CSI=0.095
threshold=0.50  TP=7 FP=76 FN=0  CSI=0.084
threshold=0.40  TP=7 FP=96 FN=0  CSI=0.068
threshold=0.30  TP=7 FP=139 FN=0  CSI=0.048


In [7]:
def hybrid_alert(rainfall_today, lgbm_proba, persistence_threshold=64.5, prob_threshold=0.5):
    return int((rainfall_today >= persistence_threshold) or (lgbm_proba >= prob_threshold))

y_pred_hybrid = np.array([
    hybrid_alert(r, p) for r, p in zip(test["rainfall_mm"], y_proba)
])
tp = ((y_test == 1) & (y_pred_hybrid == 1)).sum()
fp = ((y_test == 0) & (y_pred_hybrid == 1)).sum()
fn = ((y_test == 1) & (y_pred_hybrid == 0)).sum()
print(f"Hybrid: TP={tp} FP={fp} FN={fn} CSI={csi(y_test, y_pred_hybrid):.3f}")

Hybrid: TP=3 FP=17 FN=4 CSI=0.125


In [8]:
# Final rainfall risk function — save this, it's what the API will call
def rainfall_risk(rainfall_today_mm, lgbm_model, feature_row):
    alert = int(rainfall_today_mm >= 64.5)
    risk_score = float(lgbm_model.predict_proba(feature_row)[:, 1][0])
    return {"heavy_rain_alert": alert, "ml_risk_score": risk_score}

In [9]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)
joblib.dump(model, "../models/rainfall_lgbm.pkl")
print("Saved model")

Saved model
